## Feature Engineering

## 1. Introduction

The objective of this notebook is to transform the cleaned transaction-level dataset into a customer-level dataset using Recency, Frequency, and Monetary (RFM) analysis.

RFM analysis is a customer segmentation technique that evaluates purchasing behaviour based on three key metrics:

- **Recency (R):** Number of days since the customer's most recent purchase.
- **Frequency (F):** Number of distinct invoices associated with the customer.
- **Monetary (M):** Total purchase value generated by the customer.

These features will be used as inputs for clustering algorithms to identify groups of customers with similar purchasing behaviour.

Import the necessary libraries and load the dataset. Then, perform RFM analysis to segment customers based on their purchasing behavior. Calculate the recency, frequency, and monetary values for each customer.

Import the necessary libraries

In [1]:
import pandas as pd
import numpy as np

## 2. Loading the Cleaned Dataset

The cleaned transaction-level dataset generated in the previous notebook is loaded for further processing. The `InvoiceDate` column is converted into datetime format to enable date-based calculations.

Each row represents an individual product line within an invoice.

In [2]:
data = pd.read_csv("../data/Processed/online_retail_cleaned.csv")

## 3. Calculating Total Purchase Value

The total purchase value for each transaction line is calculated by multiplying the quantity of products purchased by their unit price.

\[
TotalPrice = Quantity \times UnitPrice
\]

This additional column will be used to calculate the Monetary value for each customer.

In [5]:
data["TotalPrice"] = data["Quantity"] * data["UnitPrice"]
data.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


## 4. Setting the Reference Date

The reference date is defined as one day after the latest transaction date in the cleaned dataset.

This date acts as the common reference point for calculating customer Recency. Using the day after the latest transaction ensures that the most recent purchase has a Recency value of at least one day.

In [8]:
data['InvoiceDate']  = pd.to_datetime(data['InvoiceDate'])
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 392692 entries, 0 to 392691
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    392692 non-null  int64         
 1   StockCode    392692 non-null  str           
 2   Description  392692 non-null  str           
 3   Quantity     392692 non-null  int64         
 4   InvoiceDate  392692 non-null  datetime64[us]
 5   UnitPrice    392692 non-null  float64       
 6   CustomerID   392692 non-null  float64       
 7   Country      392692 non-null  str           
 8   TotalPrice   392692 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(2), str(3)
memory usage: 27.0 MB


In [13]:
latest_date = data["InvoiceDate"].max()
print("Last Transtion Date: ",latest_date)
reference_date = latest_date + pd.Timedelta(days=1)
print("Reference Transtion Date: ",reference_date)


Last Transtion Date:  2011-12-09 12:50:00
Reference Transtion Date:  2011-12-10 12:50:00


## 5. Calculating Recency

Recency measures how recently a customer has made a purchase.

For each customer, the latest invoice date is identified and compared with the reference date.

\[
Recency = ReferenceDate - LastPurchaseDate
\]

A lower Recency value indicates that the customer has purchased more recently, while a higher value indicates a longer period since their last purchase.

In [25]:
recency = data.groupby("CustomerID")["InvoiceDate"].max().reset_index()
recency["Recency"] = (reference_date - recency["InvoiceDate"]).dt.days
recency.info()
recency = recency[["CustomerID","Recency"]]
recency.head()

<class 'pandas.DataFrame'>
RangeIndex: 4338 entries, 0 to 4337
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   CustomerID   4338 non-null   float64       
 1   InvoiceDate  4338 non-null   datetime64[us]
 2   Recency      4338 non-null   int64         
dtypes: datetime64[us](1), float64(1), int64(1)
memory usage: 101.8 KB


,CustomerID,Recency
0,12346.0,326
1,12347.0,2
2,12348.0,75
3,12349.0,19
4,12350.0,310


## 6. Calculating Frequency

Frequency represents the number of distinct invoices associated with each customer.

The number of unique invoice numbers is counted for every customer using the `nunique()` function.

\[
Frequency = Number\ of\ Distinct\ Invoices
\]

Distinct invoices are counted instead of transaction rows because a single invoice may contain multiple product lines. Therefore, counting rows would overestimate the number of purchase occasions.

In [59]:
frequency = data.groupby("CustomerID")["InvoiceNo"].nunique().reset_index().rename(columns={"InvoiceNo":"Frequency"})
# frequency.head()
# frequency.rename(columns={"InvoiceNo":"Frequency"},inplace=True)
frequency.head()

,CustomerID,Frequency
0,12346.0,1
1,12347.0,7
2,12348.0,4
3,12349.0,1
4,12350.0,1


## 7. Calculating Monetary

Monetary represents the total purchase value generated by each customer.

It is calculated by summing the `TotalPrice` values of all retained transaction lines belonging to each customer.

\[
Monetary = \sum (Quantity \times UnitPrice)
\]

A higher Monetary value indicates a greater total purchase value within the retained transaction data.

In [61]:
monetary = data.groupby("CustomerID")["TotalPrice"].sum().reset_index().rename(columns={"TotalPrice":"Monetary"})
# monetary.rename(columns={"TotalPrice":"Monetary"},inplace=True)
monetary.head()

,CustomerID,Monetary
0,12346.0,77183.60
1,12347.0,4310.00
2,12348.0,1797.24
3,12349.0,1757.55
4,12350.0,334.40


In [62]:
rfm = recency.merge(frequency,on="CustomerID")
rfm = rfm.merge(monetary,on="CustomerID")

rfm.head()


,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,1,77183.60
1,12347.0,2,7,4310.00
2,12348.0,75,4,1797.24
3,12349.0,19,1,1757.55
4,12350.0,310,1,334.40


## 9. RFM Dataset Validation

The generated RFM dataset is inspected to verify its structure and data quality.

The following checks are performed:

- Total number of customer records.
- Number of unique customers.
- Missing values in the RFM features.
- Duplicate customer records.
- Descriptive statistics of Recency, Frequency, and Monetary.

These checks ensure that each customer has a valid and unique RFM representation before proceeding to preprocessing and clustering.

In [63]:
rfm.info()

<class 'pandas.DataFrame'>
RangeIndex: 4338 entries, 0 to 4337
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CustomerID  4338 non-null   float64
 1   Recency     4338 non-null   int64  
 2   Frequency   4338 non-null   int64  
 3   Monetary    4338 non-null   float64
dtypes: float64(2), int64(2)
memory usage: 135.7 KB


In [70]:
rfm["CustomerID"] = rfm["CustomerID"].astype("int64")

In [64]:
rfm.shape

(4338, 4)

In [65]:
rfm["CustomerID"].nunique()

4338

In [66]:
rfm.isnull().sum()

CustomerID    0
Recency       0
Frequency     0
Monetary      0
dtype: int64

In [67]:
rfm["CustomerID"].duplicated().sum()

np.int64(0)

In [68]:
rfm.describe()

,CustomerID,Recency,Frequency,Monetary
count,4338.000000,4338.000000,4338.000000,4338.000000
mean,15300.408022,92.536422,4.272015,2048.688081
std,1721.808492,100.014169,7.697998,8985.230220
min,12346.000000,1.000000,1.000000,3.750000
25%,13813.250000,18.000000,1.000000,306.482500
50%,15299.500000,51.000000,2.000000,668.570000
75%,16778.750000,142.000000,5.000000,1660.597500
max,18287.000000,374.000000,209.000000,280206.020000


## 10. Exporting the RFM Dataset

The final customer-level RFM dataset is exported as a CSV file named `customer_rfm.csv` in the processed data directory.

This file will be used in the next stage for feature transformation, scaling, visualization, and customer clustering.

In [69]:
rfm.to_csv("../data/Processed/customer_rfm.csv",index=False)
print("RFM Dataset created sucessfully")

RFM Dataset created sucessfully
